# 04. 중계 대본 템플릿

03의 이벤트 시퀀스 → 캐스터 발화(utterance) 리스트.

- 이벤트마다 **판정 발화(call)** 를 하나씩 만든다. 나중에 TTS 타임스탬프와 정답을 잇는 고리가 `event_idx`.
- 카운트 숫자는 **재언급 발화(restate)** 에서만 말한다. (나중에 LLM 의역 단계에서 숫자를 금지하기 위함)
- 숫자는 TTS가 잘못 읽기 쉬워서 카운트는 한글 단어로 쓴다. (`2볼` → "이볼"로 읽힘, `투볼`이 맞음)
- 같은 클립은 항상 같은 대본이 나오도록 클립 ID로 난수 시드를 고정한다.

아직 없는 것: LLM 의역, 판정 생략(K1), 회상(K3), 잡담(K4) → 다음 단계

In [19]:
import json
import random
import zlib

# 03에서 저장한 파일. 없으면 03을 먼저 실행 (Colab VM이 초기화되면 사라짐)
LABELS = "labels_all.jsonl"

with open(LABELS, encoding="utf-8") as f:
    clips = [json.loads(line) for line in f]
by_id = {c["clip_id"]: c for c in clips}
print(len(clips), "클립")

46787 클립


In [20]:
TEAM = {
    "HT": "KIA", "SK": "SSG", "OB": "두산", "LG": "LG", "LT": "롯데",
    "SS": "삼성", "NC": "NC", "KT": "KT", "HH": "한화", "WO": "키움",
}
NUM = ["노", "원", "투", "쓰리"]
OUT = ["노아웃", "원아웃", "투아웃"]


def count_words(b, s, rng):
    """실제 중계에서 가장 많이 쓰는 '투앤원' 형식 위주 (04b 집계 기준). 볼을 먼저 말한다"""
    if (b, s) == (3, 2):
        core = "풀카운트"
    elif b == 0:
        core = f"노볼 {NUM[s]} 스트라이크"
    elif s == 0:
        core = f"{NUM[b]}볼"
    elif rng.random() < 0.7:
        core = f"{NUM[b]}앤{NUM[s]}"
    else:
        core = f"{NUM[b]}볼 {NUM[s]} 스트라이크"
    return rng.choice(["{}", "{}입니다", "카운트 {}", "볼카운트 {}"]).format(core)


def ordinal(n):
    return "초구" if n == 1 else f"{n}구"

In [ ]:
# 04b에서 모은 실제 중계 표현 기반. {adj} 같은 자리는 아래 FILL에서 무작위로 채운다
# (위치는 실제 투구 위치가 아니라 무작위 → 말투용 장식일 뿐, 정답과 무관)
FILL = {
    "adj":   ["바깥쪽", "몸쪽", "높은", "낮은"],
    "adv":   ["바깥쪽", "몸쪽", "높게", "낮게"],
    "side":  ["오른쪽", "왼쪽", "뒤로 가는"],
    "field": ["왼쪽", "오른쪽", "좌중간", "우중간", "중견수 앞"],
    "gap":   ["좌중간", "우중간", "좌익선상", "우익선상"],
    "pos":   ["유격수가", "2루수가", "3루수가", "1루수가", "좌익수가", "중견수가", "우익수가"],
}


def pick(options, rng):
    return rng.choice(options).format(**{k: rng.choice(v) for k, v in FILL.items()})


# 투구 판정 (타석이 이어지는 경우)
# 구종은 따로 무작위로 붙으므로 '떨어지는 공'처럼 구종을 암시하는 표현은 쓰지 않는다 (직구와 충돌)
CALL = {
    "B": ["볼", "볼이고요", "볼입니다", "{adj} 볼이고요", "{adj} 볼입니다",
          "{adv} 빠졌고요", "{adv} 빠집니다", "일단 빠집니다", "골라냅니다", "{adj} 공 골라냅니다"],
    "T": ["스트라이크", "스트라이크입니다", "{adj} 스트라이크", "{adj} 스트라이크입니다",
          "루킹 스트라이크", "그대로 들어오면서 스트라이크"],
    "S": ["헛스윙", "헛스윙이고요", "방망이 헛돕니다", "헛스윙 나옵니다", "{adj} 공에 헛스윙"],
    "F": ["파울", "파울이고요", "파울이 됐고요", "{side} 파울입니다", "먹혔고 파울입니다",
          "빗맞으면서 파울", "커트해냅니다"],
    "V": ["번트 대다가 헛스윙"],
    "W": ["번트 파울"],
    # 쳤는데 타석 결과가 없는 경우: 그 플레이에서 주자가 아웃돼 이닝 종료 (뒤에 runner_out이 이어짐)
    "H": ["쳤습니다! 타구 굴러갑니다", "쳤습니다, 타구 떠오릅니다"],
}

# 타석 종료
STRIKEOUT = {
    "T": ["루킹 삼진!", "{adj} 스트라이크, 삼진입니다", "그대로 서서 삼진 당합니다"],
    "S": ["헛스윙 삼진!", "{adj} 공에 헛스윙, 삼진입니다", "방망이 헛돌면서 삼진"],
    "V": ["번트 헛스윙, 삼진"],
    "W": ["쓰리번트 파울, 삼진입니다"],
}
IN_PLAY = {
    "single": ["안타입니다", "{field} 안타", "좋은 코스로 빠져나가는 안타입니다", "쳤습니다! 안타!"],
    "double": ["{gap} 2루타!", "{gap} 빠집니다, 2루타", "쳤습니다! 2루타!"],
    "triple": ["{gap} 3루타!", "쳤습니다! 3루타!"],
    "home_run": ["큰 타구! 넘어갑니다! 홈런!", "담장 넘어갑니다, 홈런입니다!"],
    "field_out": ["잡아냅니다, 아웃", "땅볼이고요, 잡아서 처리합니다", "뜬공, 잡았습니다",
                  "{pos} 처리합니다", "평범한 타구, 아웃"],
    "double_play": ["땅볼, 병살로 연결됩니다", "더블 플레이!"],
    "triple_play": ["트리플 플레이! 삼중살입니다"],
    "sac_fly": ["외야 뜬공, 희생플라이"],
    "sac_bunt": ["번트, 희생번트 성공합니다"],
    "field_error": ["쳤습니다, 아 실책! 타자 살아나갑니다"],
    "fielders_choice": ["땅볼, 야수선택으로 타자 1루"],
    "catcher_interference": ["포수 타격방해, 타자 1루로"],
}
WALK = ["볼넷입니다", "볼넷이네요", "볼넷으로 걸어 나갑니다", "볼넷, 1루로 걸어 나갑니다"]
HBP = ["아, 몸에 맞는 공", "몸쪽 공, 맞았습니다. 사구"]

# 투구 없는 이벤트
NO_PITCH = {
    "auto_ball": ["피치클락 위반, 볼이 하나 주어집니다", "투수 피치클락 위반, 볼"],
    "auto_strike": ["타자 피치클락 위반, 스트라이크 선언됩니다"],
    "ibb": ["고의4구, 1루로 내보냅니다"],
    "walk_award": ["1루로 걸어 나갑니다"],
    "runner_out": ["주자 아웃! 이대로 이닝 종료됩니다"],
}

In [22]:
# 발화 노브 기본값. 나중에 Easy/Hard 프리셋은 이 값들을 덮어써서 만든다
KNOBS = {
    "p_restate": 0.3,   # 판정 뒤 카운트 재언급 (K2)
    "p_ja": 0.25,       # 판정 앞 '자,'
    "p_first": 0.3,     # '초구' 언급 (실제 중계에서 초구는 가끔 말함)
    "p_ordinal": 0.05,  # '3구' 같은 2구 이후 순서 언급 (실제로는 거의 안 함, 04b 집계)
    "p_pitch": 0.3,     # 구종 언급
    "p_speed": 0.5,     # 구종을 말할 때 구속도 붙일 확률
}


def call_text(e, rng, k):
    """이벤트 하나 → 판정 발화 문장"""
    if e["type"] != "pitch":
        text = pick(NO_PITCH[e["type"]], rng)
        if e["type"] == "auto_strike" and e.get("pa_end") == "strikeout":
            text += ", 삼진"
        return text

    r, end = e["result"], e.get("pa_end")
    if end is None:
        verdict = pick(CALL[r], rng)
    elif r == "H":
        verdict = pick(IN_PLAY[end], rng)
    elif r == "HBP":
        verdict = pick(HBP, rng)
    elif end == "walk":
        verdict = pick(WALK, rng)
    elif end == "strikeout":
        verdict = pick(STRIKEOUT[r], rng)
    else:  # 드문 조합 (예: 낫아웃 후 1루 송구 아웃)
        verdict = pick(CALL[r], rng) + ", " + pick(IN_PLAY.get(end, ["아웃"]), rng)

    # 앞부분은 대부분 생략: "바깥쪽 빠졌고요" / "초구, 볼" / "슬라이더, 볼" / "131킬로 슬라이더, 볼"
    head = []
    p_order = k["p_first"] if e["pitch"] == 1 else k["p_ordinal"]
    if rng.random() < p_order:
        head.append(ordinal(e["pitch"]))
    if e["pitch_name"] and rng.random() < k["p_pitch"]:
        if e["speed"] and rng.random() < k["p_speed"]:
            head.append(f"{e['speed']}킬로")
        head.append(e["pitch_name"])
    return f"{' '.join(head)}, {verdict}" if head else verdict


def end_sentence(text):
    """이미 '!'나 '.'로 끝나면 그대로, 아니면 마침표를 붙인다"""
    return text if text.endswith(("!", ".")) else text + "."


def render_clip(clip, knobs=None):
    """클립 → 발화 리스트. 각 발화: {kind, event_idx, text}"""
    k = {**KNOBS, **(knobs or {})}
    rng = random.Random(zlib.crc32(clip["clip_id"].encode()))
    away, home = TEAM.get(clip["away"], clip["away"]), TEAM.get(clip["home"], clip["home"])
    batting = away if clip["half"] == "top" else home
    sc = clip["score"]

    utts = [{
        "kind": "intro", "event_idx": None,
        "text": f"{clip['inning']}회{'초' if clip['half'] == 'top' else '말'}, {batting} 공격입니다. "
                f"스코어 {away} {sc['away']} 대 {home} {sc['home']}.",
    }]

    pitcher, ab, outs = None, None, 0
    for i, e in enumerate(clip["events"]):
        if e["pitcher"] != pitcher:
            verb = "마운드에는" if pitcher is None else "투수 교체, 마운드에"
            utts.append({"kind": "pitcher", "event_idx": None, "text": f"{verb} {e['pitcher']}."})
            pitcher = e["pitcher"]
        if e["ab"] != ab:
            utts.append({"kind": "batter", "event_idx": None, "text": f"{OUT[outs]}, 타석에는 {e['batter']}."})
            ab = e["ab"]

        text = call_text(e, rng, k)
        if rng.random() < k["p_ja"]:  # 실제 중계에서 자주 붙는 '자,'
            text = "자, " + text
        utts.append({"kind": "call", "event_idx": i, "text": end_sentence(text)})

        if "pa_end" in e:
            outs = e["o"]
        elif e.get("result") != "H" and (e["b"], e["s"]) != (0, 0) and rng.random() < k["p_restate"]:
            # 타구 직후에는 카운트를 다시 말하지 않음
            utts.append({"kind": "restate", "event_idx": i, "text": end_sentence(count_words(e["b"], e["s"], rng))})

    # 마지막 아웃은 다음 타자 소개가 없어서 따로 말해준다 (주루사 멘트에는 이미 '이닝 종료'가 있음)
    last = clip["events"][-1]
    if last["o"] == 3 and last["type"] != "runner_out":
        utts.append({"kind": "inning_end", "event_idx": len(clip["events"]) - 1, "text": "쓰리아웃, 공수 교대합니다."})

    return utts

In [23]:
def show(clip):
    print(f"===== {clip['clip_id']}  tags={clip['tags']}")
    for u in render_clip(clip):
        idx = "" if u["event_idx"] is None else u["event_idx"]
        print(f"{u['kind']:>8} {idx:>3} | {u['text']}")
    print()


show(by_id["20260328HTSK02026_01T"])                            # 01에서 본 2026 개막전 1회초
show(next(c for c in clips if "pitch_clock" in c["tags"] and c["season"] >= 2025))
show(next(c for c in clips if "hbp" in c["tags"]))

===== 20260328HTSK02026_01T  tags=[]
   intro     | 1회초, KIA 공격입니다. 스코어 KIA 0 대 SSG 0.
 pitcher     | 마운드에는 화이트.
  batter     | 노아웃, 타석에는 김호령.
    call   0 | 자, 볼.
    call   1 | 자, 파울이고요.
    call   2 | 자, 스위퍼, 볼이고요.
    call   3 | 151킬로 직구, 떨어지는 공에 헛스윙.
    call   4 | 자, 일단 빠집니다.
    call   5 | 빗맞으면서 파울.
    call   6 | 자, 오른쪽 파울입니다.
    call   7 | 자, 볼넷, 1루로 걸어 나갑니다.
  batter     | 노아웃, 타석에는 카스트로.
    call   8 | 자, 직구, 낮은 볼이고요.
 restate   8 | 카운트 원볼.
    call   9 | 높은 공 골라냅니다.
    call  10 | 우중간 빠집니다, 2루타.
  batter     | 노아웃, 타석에는 김도영.
    call  11 | 커터, 떨어지는 공에 헛스윙.
    call  12 | 2구, 낮은 볼입니다.
    call  13 | 자, 볼입니다.
 restate  13 | 볼카운트 투앤원.
    call  14 | 평범한 타구, 아웃.
  batter     | 원아웃, 타석에는 나성범.
    call  15 | 먹혔고 파울입니다.
 restate  15 | 노볼 원 스트라이크입니다.
    call  16 | 126킬로 커브, 볼입니다.
 restate  16 | 카운트 원앤원.
    call  17 | 자, 150킬로 직구, 좋은 코스로 빠져나가는 안타입니다.
  batter     | 원아웃, 타석에는 김선빈.
    call  18 | 초구, 볼입니다.
 restate  18 | 원볼입니다.
    call  19 | 커터, 일단 빠집니다.
    call  20 | 평범한 타구, 아웃.

In [24]:
# 풀이 가능성 검사: 모든 이벤트가 판정 발화(call)를 하나씩 가지는가?
# (지금은 당연히 통과. 나중에 판정 생략 K1을 넣으면 이 검사가 의미를 가진다)
missing = 0
for c in clips:
    called = {u["event_idx"] for u in render_clip(c) if u["kind"] == "call"}
    missing += len(c["events"]) - len(called)
print("판정 발화 없는 이벤트:", missing)

판정 발화 없는 이벤트: 0
